In [2]:
!pip install torch torchvision torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.0 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
def image_to_graph(img, label):
    img = img.view(28, 28)

    xs = []
    for i in range(28):
        for j in range(28):
            intensity = img[i, j]

            x_pos = i / 27.0
            y_pos = j / 27.0

            dx = img[i, j] - img[i, j-1] if j > 0 else 0
            dy = img[i, j] - img[i-1, j] if i > 0 else 0

            xs.append([intensity, x_pos, y_pos, dx, dy])

    x = torch.tensor(xs, dtype=torch.float)

    # normalize features
    x = (x - x.mean(dim=0)) / (x.std(dim=0) + 1e-6)

    # edges (4-neighbour)
    edges = []
    for i in range(28):
        for j in range(28):
            node = i * 28 + j

            for di, dj in [(1,0), (-1,0), (0,1), (0,-1)]:
                ni, nj = i + di, j + dj

                if 0 <= ni < 28 and 0 <= nj < 28:
                    neighbor = ni * 28 + nj
                    edges.append([node, neighbor])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index, y=torch.tensor(label))

In [5]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# smaller subset for speed (you can increase later)
train_graphs = [image_to_graph(train_dataset[i][0], train_dataset[i][1]) for i in range(5000)]
test_graphs  = [image_to_graph(test_dataset[i][0], test_dataset[i][1]) for i in range(1000)]

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_graphs, batch_size=32)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.55MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.43MB/s]


In [6]:
class GraphSAGE(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = SAGEConv(5, 128)
        self.conv2 = SAGEConv(128, 256)
        self.conv3 = SAGEConv(256, 256)

        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(256)

        self.fc = nn.Linear(512, 10)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Layer 1
        x1 = self.conv1(x, edge_index)
        x1 = self.bn1(x1)
        x1 = F.relu(x1)
        x1 = F.dropout(x1, p=0.3, training=self.training)

        # Layer 2 + residual
        x2 = self.conv2(x1, edge_index)
        x2 = self.bn2(x2)
        x2 = F.relu(x2)
        x2 = x2 + x1.repeat(1, 2)[:, :256]  # simple residual match
        x2 = F.dropout(x2, p=0.3, training=self.training)

        # Layer 3
        x3 = self.conv3(x2, edge_index)
        x3 = self.bn3(x3)
        x3 = F.relu(x3)

        # Pooling
        x_mean = global_mean_pool(x3, batch)
        x_max  = global_max_pool(x3, batch)

        x = torch.cat([x_mean, x_max], dim=1)

        return self.fc(x)

In [7]:
model = GraphSAGE().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5
)

In [8]:
def train():
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)

        optimizer.zero_grad()
        out = model(data)

        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss


@torch.no_grad()
def test(loader):
    model.eval()
    correct = 0

    for data in loader:
        data = data.to(device)

        out = model(data)
        pred = out.argmax(dim=1)

        correct += int((pred == data.y).sum())

    return correct / len(loader.dataset)

In [9]:
epochs = 30

for epoch in range(1, epochs + 1):
    loss = train()
    acc = test(test_loader)

    scheduler.step()

    print(f"Epoch {epoch:02d} | Loss {loss:.4f} | Test Acc {acc:.4f}")

Epoch 01 | Loss 261.4174 | Test Acc 0.6800
Epoch 02 | Loss 117.6042 | Test Acc 0.8260
Epoch 03 | Loss 68.2656 | Test Acc 0.9130
Epoch 04 | Loss 43.8567 | Test Acc 0.9270
Epoch 05 | Loss 30.3661 | Test Acc 0.9330
Epoch 06 | Loss 25.7076 | Test Acc 0.9400
Epoch 07 | Loss 21.8464 | Test Acc 0.9630
Epoch 08 | Loss 19.9958 | Test Acc 0.9670
Epoch 09 | Loss 16.7910 | Test Acc 0.9640
Epoch 10 | Loss 17.4312 | Test Acc 0.9590
Epoch 11 | Loss 10.3772 | Test Acc 0.9760
Epoch 12 | Loss 10.1751 | Test Acc 0.9790
Epoch 13 | Loss 10.1321 | Test Acc 0.9780
Epoch 14 | Loss 9.8360 | Test Acc 0.9760
Epoch 15 | Loss 10.1624 | Test Acc 0.9790
Epoch 16 | Loss 9.0386 | Test Acc 0.9760
Epoch 17 | Loss 8.6774 | Test Acc 0.9730
Epoch 18 | Loss 7.5148 | Test Acc 0.9770
Epoch 19 | Loss 7.0935 | Test Acc 0.9740
Epoch 20 | Loss 7.9282 | Test Acc 0.9760
Epoch 21 | Loss 6.5294 | Test Acc 0.9740
Epoch 22 | Loss 6.6472 | Test Acc 0.9800
Epoch 23 | Loss 5.4811 | Test Acc 0.9800
Epoch 24 | Loss 5.8579 | Test Acc 0.9780
